# Backfill `created_at` Through Source Timeline

This notebook updates `created_at` values to simulate temporal ingestion:

1. `(:Episodic)` nodes: set from `data/sources.json` via `source_text_name -> date_accessed`.
2. Non-episodic nodes: set to earliest `created_at` among connected episodic nodes.
3. Edges: set to earliest `created_at` among episodic nodes referenced by relationship `episodes` (UUID list).


In [1]:
import json
from pathlib import Path
from datetime import datetime

from neo4j import GraphDatabase

# Neo4j connection
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "knowledgegraph"
NEO4J_DATABASE = "neo4j"

# Data source metadata
SOURCES_PATH = Path("data/sources.json")


In [2]:
# ---- Build source_text_name -> Neo4j datetime string mapping ----
with SOURCES_PATH.open("r", encoding="utf-8") as f:
    sources = json.load(f)

entries = sources.get("entries", [])
assert entries, "No entries found in data/sources.json"

def date_accessed_to_neo4j_datetime(date_str: str) -> str:
    # Input: YYYY-MM-DD
    # Output: YYYY-MM-DDT00:00:00.000000000Z
    d = datetime.strptime(date_str, "%Y-%m-%d").date()
    return f"{d.isoformat()}T00:00:00.000000000Z"

source_rows = []
for e in entries:
    source_rows.append(
        {
            "source_text_name": e["file_name"],
            "created_at_str": date_accessed_to_neo4j_datetime(e["date_accessed"]),
        }
    )

print("Source mapping rows:")
for r in source_rows:
    print(r)


Source mapping rows:
{'source_text_name': 'haaland_2019', 'created_at_str': '2019-12-26T00:00:00.000000000Z'}
{'source_text_name': 'transfermarkt_2019', 'created_at_str': '2019-12-29T00:00:00.000000000Z'}
{'source_text_name': 'transfermarkt_2022', 'created_at_str': '2022-06-13T00:00:00.000000000Z'}


In [3]:
# ---- Connect to Neo4j ----
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

with driver.session(database=NEO4J_DATABASE) as session:
    counts = session.run(
        """
        MATCH (ep:Episodic)
        RETURN count(ep) AS episodic_count
        """
    ).single()

print("Episodic nodes in DB:", counts["episodic_count"])


Episodic nodes in DB: 16


In [4]:
# ---- Step 1: Update Episodic.created_at from source_text_name ----
# This requires ep.source_text_name to already be populated.
update_ep_q = """
UNWIND $rows AS row
MATCH (ep:Episodic {source_text_name: row.source_text_name})
SET ep.created_at = datetime(row.created_at_str)
RETURN count(ep) AS updated
"""

with driver.session(database=NEO4J_DATABASE) as session:
    updated_ep = session.run(update_ep_q, rows=source_rows).single()["updated"]

print("Updated Episodic created_at:", updated_ep)

# Quick diagnostic for episodic nodes that still miss source_text_name or created_at
diag_q = """
MATCH (ep:Episodic)
RETURN
  count(ep) AS total,
  count(CASE WHEN ep.source_text_name IS NULL THEN 1 END) AS missing_source_text_name,
  count(CASE WHEN ep.created_at IS NULL THEN 1 END) AS missing_created_at
"""
with driver.session(database=NEO4J_DATABASE) as session:
    diag = session.run(diag_q).single()

print(dict(diag))


Updated Episodic created_at: 16
{'total': 16, 'missing_source_text_name': 0, 'missing_created_at': 0}


In [5]:
# ---- Step 2: Update non-Episodic nodes from earliest connected Episodic.created_at ----
# Uses any connection pattern n--(ep:Episodic).
update_nodes_q = """
MATCH (n)
WHERE NOT n:Episodic
OPTIONAL MATCH (n)--(ep:Episodic)
WITH n, min(ep.created_at) AS earliest
WHERE earliest IS NOT NULL
SET n.created_at = earliest
RETURN count(n) AS updated
"""

with driver.session(database=NEO4J_DATABASE) as session:
    updated_nodes = session.run(update_nodes_q).single()["updated"]

print("Updated non-Episodic node created_at:", updated_nodes)


Updated non-Episodic node created_at: 51


In [6]:
# ---- Step 3: Update edge created_at from earliest Episodic.created_at via r.episodes UUID list ----
update_edges_q = """
MATCH ()-[r]->()
WHERE r.episodes IS NOT NULL AND size(r.episodes) > 0
UNWIND r.episodes AS ep_uuid
MATCH (ep:Episodic {uuid: ep_uuid})
WITH r, min(ep.created_at) AS earliest
WHERE earliest IS NOT NULL
SET r.created_at = earliest
RETURN count(r) AS updated
"""

with driver.session(database=NEO4J_DATABASE) as session:
    updated_edges = session.run(update_edges_q).single()["updated"]

print("Updated edge created_at:", updated_edges)


Updated edge created_at: 42


In [7]:
# ---- Verification ----
verify_q = """
CALL {
  MATCH (ep:Episodic)
  RETURN
    count(ep) AS total_ep,
    count(CASE WHEN ep.created_at IS NULL THEN 1 END) AS ep_missing_created
}
CALL {
  MATCH (n)
  WHERE NOT n:Episodic
  RETURN
    count(n) AS total_non_ep,
    count(CASE WHEN n.created_at IS NULL THEN 1 END) AS non_ep_missing_created
}
CALL {
  MATCH ()-[r]->()
  RETURN
    count(r) AS total_rel,
    count(CASE WHEN r.created_at IS NULL THEN 1 END) AS rel_missing_created,
    count(CASE WHEN r.episodes IS NOT NULL AND size(r.episodes) > 0 THEN 1 END) AS rel_with_episodes
}
RETURN *
"""

with driver.session(database=NEO4J_DATABASE) as session:
    summary = session.run(verify_q).single()

print(dict(summary))


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nCALL {\n  MATCH (ep:Episodic)\n  RETURN\n    count(ep) AS total_ep,\n    count(CASE WHEN ep.created_at IS NULL THEN 1 END) AS ep_missing_created\n}\nCALL {\n  MATCH (n)\n  WHERE NOT n:Episodic\n  RETURN\n    count(n) AS total_non_ep,\n    count(CASE WHEN n.created_at IS NULL THEN 1 END) AS non_ep_missing_created\n}\nCALL {\n  MATCH ()-[r]->()\

{'ep_missing_created': 0, 'non_ep_missing_created': 0, 'rel_missing_created': 0, 'rel_with_episodes': 42, 'total_ep': 16, 'total_non_ep': 51, 'total_rel': 147}


In [ ]:
    # ---- Optional: sample rows for sanity check ----
    sample_nodes_q = """
    MATCH (n)
    RETURN labels(n) AS labels, n.name AS name, n.uuid AS uuid, n.created_at AS created_at
    ORDER BY n.created_at ASC
    LIMIT 20
    """

    sample_edges_q = """
    MATCH ()-[r]->()
    RETURN type(r) AS type, r.name AS name, r.uuid AS uuid, r.created_at AS created_at, r.episodes AS episodes
    ORDER BY r.created_at ASC
    LIMIT 20
    """

    with driver.session(database=NEO4J_DATABASE) as session:
        sample_nodes = list(session.run(sample_nodes_q))
        sample_edges = list(session.run(sample_edges_q))

    print("Sample nodes:")
    for row in sample_nodes:
        print(dict(row))

    print("
Sample edges:")
    for row in sample_edges:
        print(dict(row))


In [8]:
driver.close()
print("Done.")


Done.
